In [1]:
from evidently import DataDefinition, Report, Dataset
from evidently.metrics import QuantileValue, MissingValueCount, UniqueValueCount, AlmostConstantColumnsCount
import pandas as pd


## Q1. Prepare the dataset

In [2]:
file_name = "data/green_tripdata_2024-03.parquet"

In [3]:
def read_file(file_name):
    df = pd.read_parquet(file_name)
    df = df[(df['lpep_pickup_datetime'] >= '2024-03-01') & (df['lpep_pickup_datetime'] < '2024-04-01')]
    return df
df = read_file(file_name)
print(f"number of rows: {len(df)}")

number of rows: 57447


In [4]:
categorical_cols = [
    'RatecodeID', 
    'passenger_count', 
    'payment_type', 
    'trip_type', 
    'congestion_surcharge',
    'store_and_fwd_flag' # Good to cast this too, as it also has NaNs
]

df[categorical_cols] = df[categorical_cols].astype("str")
eval_data = Dataset.from_pandas(df, data_definition=DataDefinition())

/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/evidently/core/datasets.py:1760: RuntimeWarning: invalid value encountered in scalar divide
  infinite=StatCountValue(infinite_count, infinite_count / data.count()),


In [5]:
report = Report(
    metrics=[
        QuantileValue(column="fare_amount", quantile=0.5),
        # MissingValueCount(column="trip_distance"),
        # UniqueValueCount(column="trip_distance"),
        # AlmostConstantColumnsCount(column="trip_distance", threshold=0.95)
    ]
)

## Simulate Real world Production Environment of Reciving data every day

In [6]:
df['date'] = df['lpep_pickup_datetime'].dt.date
daily_groups = df.groupby('date')

In [ ]:
max_median = 0
for date, daily_df in daily_groups:
    # report.run(current_data=Dataset.from_pandas(group, data_definition=DataDefinition()))
    rep = report.run(current_data=daily_df, reference_data=None)
    result_dict = rep.dict()
    print(result_dict["metrics"][0])
    median = result_dict["metrics"][0]["current"]["value"]
    if median > max_median:
        max_median = median
    print(f"Median fare amount for date {date}: {median}\n\n")
print(f"Max median fare amount across all dates: {max_median}")

{'id': '641f3d487377ef8aeba674733f8705f4', 'metric_name': 'QuantileValue(column=fare_amount,quantile=0.5)', 'config': {'type': 'evidently:metric_v2:QuantileValue', 'column': 'fare_amount', 'quantile': 0.5}, 'value': np.float64(13.5)}


/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/evidently/core/datasets.py:1760: RuntimeWarning:

invalid value encountered in scalar divide



KeyError: 'result'

In [ ]:
# report.run(current_data=eval_data)

In [ ]:
df.info()